In [1]:
import SimpleITK as sitk
import torch
import glob
import numpy as np
import os
import scipy.spatial
import pandas as pd
from tqdm import tqdm


def get_hausdorff(test_image, result_image):
    """Compute the Hausdorff distance."""

    result_statistics = sitk.StatisticsImageFilter()
    result_statistics.Execute(result_image)

    if result_statistics.GetSum() == 0:
        hd = np.nan
        return hd

    # Edge detection is done by ORIGINAL - ERODED, keeping the outer boundaries of lesions. Erosion is performed in 3D
    e_test_image = sitk.BinaryErode(test_image, (1, 1, 1))
    e_result_image = sitk.BinaryErode(result_image, (1, 1, 1))

    h_test_image = sitk.Subtract(test_image, e_test_image)
    h_result_image = sitk.Subtract(result_image, e_result_image)

    h_test_indices = np.flip(np.argwhere(sitk.GetArrayFromImage(h_test_image))).tolist()
    h_result_indices = np.flip(
        np.argwhere(sitk.GetArrayFromImage(h_result_image))
    ).tolist()

    test_coordinates = [
        test_image.TransformIndexToPhysicalPoint(x) for x in h_test_indices
    ]
    result_coordinates = [
        test_image.TransformIndexToPhysicalPoint(x) for x in h_result_indices
    ]

    def get_distances_from_a_to_b(a, b):
        kd_tree = scipy.spatial.KDTree(a, leafsize=100)
        return kd_tree.query(b, k=1, eps=0, p=2)[0]

    d_test_to_result = get_distances_from_a_to_b(test_coordinates, result_coordinates)
    d_result_to_test = get_distances_from_a_to_b(result_coordinates, test_coordinates)

    hd = max(np.percentile(d_test_to_result, 95), np.percentile(d_result_to_test, 95))

    return hd


def get_metrics(test_image, label_image):
    """Compute the precision and recall."""
    test_array = sitk.GetArrayFromImage(test_image).flatten()
    result_array = sitk.GetArrayFromImage(label_image).flatten()
    if np.all(test_array == 0) and np.all(result_array == 0):
        return 0, 0, 0, 0, 0, 0, 0

    true_positive = np.sum(test_array * result_array)
    false_positive = np.sum((1 - test_array) * result_array)
    false_negative = np.sum(test_array * (1 - result_array))

    sensitivity = (
        true_positive / (true_positive + false_negative)
        if (true_positive + false_negative)
        else 0
    )
    precision = (
        true_positive / (true_positive + false_positive)
        if (true_positive + false_positive)
        else 0
    )
    dice = (
        (2 * true_positive) / (2 * true_positive + false_positive + false_negative)
        if (2 * true_positive + false_positive + false_negative)
        else 0
    )

    hd95 = get_hausdorff(test_image, label_image)
    return (sensitivity, precision, dice, hd95)


def calculate(prediction_path, ground_truth_path):
    metrics_list = []

    test_list = [
        x
        for x in os.listdir(prediction_path)
        if x.endswith(".nii.gz") or x.endswith(".mha")
    ]
    for name in tqdm(sorted(test_list)):
        try:
            test_image_path = os.path.join(ground_truth_path, name)
            result_image_path = os.path.join(prediction_path, name)
            test_image = sitk.ReadImage(test_image_path)
            result_image = sitk.ReadImage(result_image_path)
            assert test_image.GetSize() == result_image.GetSize()

            # Copy meta information
            result_image.CopyInformation(test_image)

            # Apply mask for treated aneurysms
            test_image = test_image > 0.5
            result_image = result_image > 0.5

            # Compute metrics
            try:
                h95 = get_hausdorff(test_image, result_image)
            except Exception as e:
                # print(f"Error calculating Hausdorff for {name}: {e}")
                h95 = 0

            (sensitivity, precision, dice, h95) = get_metrics(test_image, result_image)

            # Append metrics for current image to list
            metrics_list.append(
                {
                    "name": name,
                    "sensitivity": sensitivity,
                    "precision": precision,
                    "dice": dice,
                    "hausdorff95": h95,
                }
            )
        except Exception as e:
            print(f"Failed to process {name}: {e}")
            continue

    # Create DataFrame from list of metrics
    info_df = pd.DataFrame(metrics_list)
    return info_df


label_path = "/data1/songwei/WorkStation/Data/raw/labelsTr"
paths = ["/data1/songwei/Segment/predictions/test"]

results = []
for p in paths:
    new_fold = (
        calculate(
            prediction_path=p,
            ground_truth_path=label_path,
        ),
    )[0]

    print(p)
    print(new_fold.describe())
    results.append(new_fold)

100%|██████████| 10/10 [00:05<00:00,  1.69it/s]

/data1/songwei/Segment/predictions/test
       sensitivity  precision  dice  hausdorff95
count         10.0       10.0  10.0          0.0
mean           0.0        0.0   0.0          NaN
std            0.0        0.0   0.0          NaN
min            0.0        0.0   0.0          NaN
25%            0.0        0.0   0.0          NaN
50%            0.0        0.0   0.0          NaN
75%            0.0        0.0   0.0          NaN
max            0.0        0.0   0.0          NaN
